# ML量化框架使用示例

本Notebook展示如何使用可插拔的机器学习模型进行量化交易策略开发。

## 1. 导入必要的模块

In [1]:
# 深度学习模型同样可以无缝替换（需要安装 torch: pip install torch）
try:
    from ml_framework.models.pytorch_models import MLPModel
    
    model_mlp = MLPModel(
        input_dim=X_train.shape[1],
        hidden_dims=[128, 64, 32],
        dropout_rate=0.3,
        epochs=50,
        batch_size=256,
        lr=0.001
    )

    history_mlp = model_mlp.fit(X_train, y_train, X_val, y_val)

    test_metrics_mlp = model_mlp.evaluate(X_test, y_test)
    print(f"\n测试集表现:")
    for k, v in test_metrics_mlp.items():
        print(f"   {k}: {v:.4f}")
except ImportError as e:
    print(f"⚠️ PyTorch 未安装，跳过 MLP: {e}")
    print("   安装命令: pip install torch")
    test_metrics_mlp = {}

## 2. 数据加载

In [2]:
loader = StockDataLoader(DATA_PATH)
df = loader.load(years_back=5)

# 选择50只样本股票
selected_codes = loader.select_sample_codes(n=1000)
df = df[df['code'].isin(selected_codes)]

print(f"股票数量: {df['code'].nunique()}")
print(f"日期范围: {df['date'].min()} ~ {df['date'].max()}")

df.head()

📂 加载数据: /Users/harry/quant-learning/ml_framework/../data/a_stock_history_price.csv
   原始: 7,783,131 行
   加载后: 6,706,561 行 | 6403 只股票
股票数量: 1000
日期范围: 2021-02-18 00:00:00 ~ 2026-02-13 00:00:00


,date,code,name,open,close,high,low,price_change,pct_change,volume,turnover_rate,market_cap,dividends,stock_splits,amount
9712,2021-02-18,000012,南玻Ａ,5.6654,5.6566,5.7890,5.6037,0.11,1.91,25525587.0,1.30,1.109442e+10,0.0,0.0,1.443880e+08
9713,2021-02-19,000012,南玻Ａ,5.6743,5.8243,5.8331,5.6213,0.17,2.96,31539389.0,1.61,1.142333e+10,0.0,0.0,1.836949e+08
9714,2021-02-22,000012,南玻Ａ,5.8331,6.1331,6.3096,5.8243,0.31,5.30,63929777.0,3.26,1.202899e+10,0.0,0.0,3.920877e+08
9715,2021-02-23,000012,南玻Ａ,6.1331,6.1331,6.2214,6.0096,0.00,0.00,46435967.0,2.37,1.202899e+10,0.0,0.0,2.847964e+08
9716,2021-02-24,000012,南玻Ａ,6.0537,5.8949,6.1684,5.8507,-0.24,-3.88,37957972.0,1.94,1.156180e+10,0.0,0.0,2.237584e+08


## 3. 数据集划分（时序划分）

In [3]:
train_df, val_df, test_df = time_series_split(df)

print(f"训练集: {len(train_df)} 行")
print(f"验证集: {len(val_df)} 行")
print(f"测试集: {len(test_df)} 行")


⏰ 时间序列数据集划分...
   训练集: 578,506 (2021-02-18 ~ 2024-02-19)
   验证集: 222,739 (2024-02-20 ~ 2025-02-20)
   测试集: 237,197 (2025-02-21 ~ 2026-02-13)
训练集: 578506 行
验证集: 222739 行
测试集: 237197 行


## 4. 特征工程

In [4]:
from sklearn.preprocessing import StandardScaler

fe = FeatureEngineer()

# 生成特征
train_features = fe.create_features(train_df)
val_features = fe.create_features(val_df)
test_features = fe.create_features(test_df)

# 标准化
scaler = StandardScaler()
X_train, y_train, _ = fe.prepare_xy(train_features, scaler, fit_scaler=True)
X_val, y_val, _ = fe.prepare_xy(val_features, scaler, fit_scaler=False)
X_test, y_test, _ = fe.prepare_xy(test_features, scaler, fit_scaler=False)

# 从 fe 对象获取特征列
feature_cols = fe.feature_cols
print(f"特征数量: {len(feature_cols)}")
print(f"特征列表: {feature_cols[:5]} ...")


🔧 构建特征...
   特征数: 40
   有效样本: 516,097

🔧 构建特征...
   特征数: 40
   有效样本: 158,308

🔧 构建特征...
   特征数: 40
   有效样本: 168,126
特征数量: 40
特征列表: ['ma_5', 'price_to_ma_5', 'ma_5_slope', 'ma_10', 'price_to_ma_10'] ...


## 5. 模型训练与评估

这里演示**可插拔**的特性：只需更换模型类，其他代码完全相同！

### 5.1 随机森林

In [5]:
# 创建模型
model_rf = RandomForestModel(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# 训练
history_rf = model_rf.fit(X_train, y_train, X_val, y_val)

# 评估
test_metrics_rf = model_rf.evaluate(X_test, y_test)
print(f"\n测试集表现:")
for k, v in test_metrics_rf.items():
    print(f"   {k}: {v:.4f}")


测试集表现:
   RMSE: 0.0615
   MAE: 0.0390
   R2: -0.0517


### 5.2 XGBoost

In [ ]:
# 更换模型，其他代码不变！
model_xgb = XGBoostModel(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1
)

history_xgb = model_xgb.fit(X_train, y_train, X_val, y_val)

test_metrics_xgb = model_xgb.evaluate(X_test, y_test)
print(f"\n测试集表现:")
for k, v in test_metrics_xgb.items():
    print(f"   {k}: {v:.4f}")

### 5.3 LightGBM

In [7]:
model_lgb = LightGBMModel(
    n_estimators=100,
    num_leaves=31,
    learning_rate=0.05
)

history_lgb = model_lgb.fit(X_train, y_train, X_val, y_val)

test_metrics_lgb = model_lgb.evaluate(X_test, y_test)
print(f"\n测试集表现:")
for k, v in test_metrics_lgb.items():
    print(f"   {k}: {v:.4f}")


测试集表现:
   RMSE: 0.0614
   MAE: 0.0389
   R2: -0.0480


### 5.4 MLP神经网络（PyTorch）

In [8]:
# 深度学习模型同样可以无缝替换（需要安装 torch: pip install torch）
try:
    from ml_framework.models.pytorch_models import MLPModel
    
    model_mlp = MLPModel(
        input_dim=X_train.shape[1],
        hidden_dims=[128, 64, 32],
        dropout_rate=0.3,
        epochs=50,
        batch_size=256,
        lr=0.001
    )

    history_mlp = model_mlp.fit(X_train, y_train, X_val, y_val)

    test_metrics_mlp = model_mlp.evaluate(X_test, y_test)
    print(f"\n测试集表现:")
    for k, v in test_metrics_mlp.items():
        print(f"   {k}: {v:.4f}")
except ImportError as e:
    print(f"⚠️ PyTorch 未安装，跳过 MLP: {e}")
    print("   安装命令: pip install torch")
    test_metrics_mlp = {}

ImportError: 请安装 torch: pip install torch

## 6. 模型对比

In [ ]:
import pandas as pd

# 收集所有可用的模型结果
model_results = {
    'RandomForest': test_metrics_rf,
    'XGBoost': test_metrics_xgb,
    'LightGBM': test_metrics_lgb,
}

# 如果 MLP 可用，也加入对比
if 'test_metrics_mlp' in dir() and test_metrics_mlp:
    model_results['MLP'] = test_metrics_mlp

comparison = pd.DataFrame(model_results)

print("\n模型对比:")
print(comparison.round(4))

## 7. 策略回测

In [ ]:
# 使用表现最好的模型进行回测
y_pred = model_xgb.predict(X_test)
test_features['pred_return'] = y_pred

backtester = Backtester(top_k=10)
result = backtester.run(test_features)

# 绘制结果
backtester.plot_results(result)

## 8. 一键运行完整流程

In [ ]:
from ml_framework.main import run_ml_pipeline

# 只需一行代码，更换模型类即可
result = run_ml_pipeline(XGBoostModel, {
    'n_estimators': 100,
    'max_depth': 5
})

# 切换到MLP模型
# result = run_ml_pipeline(MLPModel, {
#     'hidden_dims': [128, 64, 32],
#     'dropout_rate': 0.3
# })